# STAR injection on the Steane [[7, 1, 3]] color code

This notebook keeps the injection at the **physical Squin** level: one logical Steane block is represented by seven physical qubits, and the STAR/TMR layer applies physical `Rz(theta_star)` gates only to a weight-3 logical-`Z` support.

The ideal verification below is a small statevector simulator for the Steane code. It checks that postselecting the zero `X`-stabilizer syndrome after the TMR layer produces the target logical state

`|m_theta>_L = Rz_L(theta) |+>_L`.

The paper-side Steane/color-code convention in `distillation_sim` uses logical support `[4, 5, 6]`; change `STAR_LOGICAL_Z_SUPPORT` if you want to use a different equivalent logical line.

In [1]:
import cmath
import math
from typing import Iterable

# Steane [[7,1,3]] CSS checks in the same convention as distillation_sim.circuits.steane.
# They are the X-check supports used to detect Z errors after the STAR/TMR layer.
STEANE_CHECKS = ((0, 1, 2, 3), (2, 3, 4, 6), (1, 2, 4, 5))

# Weight-3 representative of logical Z_L. For this self-dual code the same
# support can also be used as a logical X_L representative in the verifier.
STAR_LOGICAL_Z_SUPPORT = (4, 5, 6)
STAR_LOGICAL_X_SUPPORT = STAR_LOGICAL_Z_SUPPORT

N = 7
DIM = 1 << N


def mask(support: Iterable[int]) -> int:
    out = 0
    for q in support:
        out ^= 1 << q
    return out


def hamming_weight(bits: int) -> int:
    return bin(bits).count("1")


def parity(bits: int, support: Iterable[int]) -> int:
    return hamming_weight(bits & mask(support)) & 1


def zero_state() -> list[complex]:
    state = [0j] * DIM
    state[0] = 1.0 + 0j
    return state


def plus_product_state() -> list[complex]:
    amp = 1 / math.sqrt(DIM)
    return [amp + 0j for _ in range(DIM)]


def norm2(state: list[complex]) -> float:
    return sum(abs(a) ** 2 for a in state)


def normalize(state: list[complex]) -> list[complex]:
    n = math.sqrt(norm2(state))
    if n == 0:
        raise ValueError("zero vector")
    return [a / n for a in state]


def inner(left: list[complex], right: list[complex]) -> complex:
    return sum(a.conjugate() * b for a, b in zip(left, right))


def add_states(*weighted_states: tuple[complex, list[complex]]) -> list[complex]:
    out = [0j] * DIM
    for weight, state in weighted_states:
        for i, amp in enumerate(state):
            out[i] += weight * amp
    return out


def apply_x_string(state: list[complex], support: Iterable[int]) -> list[complex]:
    flip = mask(support)
    return [state[i ^ flip] for i in range(DIM)]


def project_z_plus(state: list[complex], support: Iterable[int]) -> list[complex]:
    # Project onto +1 eigenspace of product Z over support.
    return [amp if parity(i, support) == 0 else 0j for i, amp in enumerate(state)]


def project_x_plus(state: list[complex], support: Iterable[int]) -> list[complex]:
    # Project with (I + product X_support)/2.
    flip = mask(support)
    return [(state[i] + state[i ^ flip]) / 2 for i in range(DIM)]


def project_zero_x_syndrome(state: list[complex]) -> list[complex]:
    accepted = state
    for check in STEANE_CHECKS:
        accepted = project_x_plus(accepted, check)
    return accepted


def steane_logical_plus() -> list[complex]:
    # |+>^7 projected into the +1 eigenspace of the Z stabilizers gives |+>_L.
    state = plus_product_state()
    for check in STEANE_CHECKS:
        state = project_z_plus(state, check)
    return normalize(state)


def steane_logical_zero() -> list[complex]:
    # |0>^7 projected into the +1 eigenspace of the X stabilizers gives |0>_L.
    state = zero_state()
    for check in STEANE_CHECKS:
        state = project_x_plus(state, check)
    return normalize(state)


def steane_logical_one() -> list[complex]:
    return normalize(apply_x_string(steane_logical_zero(), STAR_LOGICAL_X_SUPPORT))


def directly_encoded_rz_plus(theta: float) -> list[complex]:
    # Encode the bare state Rz(theta)|+> = (e^{-i theta/2}|0> + e^{i theta/2}|1>)/sqrt(2).
    logical_zero = steane_logical_zero()
    logical_one = steane_logical_one()
    return normalize(
        add_states(
            (cmath.exp(-0.5j * theta), logical_zero),
            (cmath.exp(0.5j * theta), logical_one),
        )
    )


def apply_physical_rz_layer(state: list[complex], support: Iterable[int], angle: float) -> list[complex]:
    # Rz(angle) = exp(-i angle Z/2), applied independently on each support qubit.
    return apply_tmr_z_rotations(state, [(q,) for q in support], angle)


def apply_z_string_rotation(state: list[complex], support: Iterable[int], angle: float) -> list[complex]:
    # exp(-i angle Z_support/2), where Z_support is a product of Z on all support qubits.
    out = []
    for bits, amp in enumerate(state):
        z_string_eigenvalue = 1 if parity(bits, support) == 0 else -1
        out.append(cmath.exp(-0.5j * angle * z_string_eigenvalue) * amp)
    return out


def apply_tmr_z_rotations(
    state: list[complex], z_rotation_supports: Iterable[tuple[int, ...]], angle: float
) -> list[complex]:
    out = state
    for support in z_rotation_supports:
        out = apply_z_string_rotation(out, support, angle)
    return out


def apply_logical_rx(state: list[complex], logical_x_support: Iterable[int], angle: float) -> list[complex]:
    # exp(-i angle X_L/2), with X_L represented by a physical X product.
    c = math.cos(angle / 2)
    s = math.sin(angle / 2)
    x_state = apply_x_string(state, logical_x_support)
    return [c * amp - 1j * s * x_amp for amp, x_amp in zip(state, x_state)]


def apply_logical_rz(state: list[complex], logical_z_support: Iterable[int], angle: float) -> list[complex]:
    # A logical Rz is represented by a physical Z product over a logical-Z support.
    return apply_z_string_rotation(state, logical_z_support, angle)


def star_theta_star(theta: float, k: int = 3) -> float:
    # Magnitude from tan(theta/2) = tan(theta_star/2)^k.
    # For odd k=3, the sign below makes the accepted state match Rz_L(theta)|+>_L.
    # For even k, a sign alone cannot fix the raw (-i)^k phase into a direct Rz state.
    magnitude = 2 * math.atan(abs(math.tan(theta / 2)) ** (1 / k))
    if k % 2 == 1 and k % 4 == 3:
        return -math.copysign(magnitude, theta)
    return math.copysign(magnitude, theta)


def star_inject_and_postselect(
    theta: float,
    z_rotation_supports: Iterable[tuple[int, ...]] | None = None,
):
    if z_rotation_supports is None:
        z_rotation_supports = tuple((q,) for q in STAR_LOGICAL_Z_SUPPORT)
    else:
        z_rotation_supports = tuple(z_rotation_supports)

    k = len(z_rotation_supports)
    theta_star = star_theta_star(theta, k=k)
    initial = steane_logical_plus()
    after_tmr = apply_tmr_z_rotations(initial, z_rotation_supports, theta_star)

    accepted = project_zero_x_syndrome(after_tmr)
    success_probability = norm2(accepted)
    accepted = normalize(accepted)

    direct_target = directly_encoded_rz_plus(theta)
    product_rotation_target = normalize(apply_logical_rz(initial, STAR_LOGICAL_Z_SUPPORT, theta))
    direct_target_fidelity = abs(inner(direct_target, accepted)) ** 2
    product_target_fidelity = abs(inner(product_rotation_target, accepted)) ** 2
    return (
        theta_star,
        success_probability,
        direct_target_fidelity,
        product_target_fidelity,
        accepted,
        direct_target,
    )


def ideal_star_success_probability(theta_star: float, k: int = 3) -> float:
    c = math.cos(theta_star / 2)
    s = math.sin(theta_star / 2)
    return c ** (2 * k) + s ** (2 * k)


In [2]:
# Run the ideal STAR/TMR verification for the k=3, single-qubit-Rz version.
theta = math.pi / 16
(
    theta_star,
    p_success,
    direct_target_fidelity,
    product_target_fidelity,
    accepted,
    direct_target,
) = star_inject_and_postselect(theta)

print(f"target logical theta                = {theta:.12f} rad")
print(f"physical STAR theta_star           = {theta_star:.12f} rad")
print(f"success probability                = {p_success:.12f}")
print(f"ideal STAR probability             = {ideal_star_success_probability(theta_star):.12f}")
print(f"fidelity vs direct encoded target  = {direct_target_fidelity:.16f}")
print(f"fidelity vs product-rotation check = {product_target_fidelity:.16f}")

assert abs(p_success - ideal_star_success_probability(theta_star)) < 1e-12
assert direct_target_fidelity > 1 - 1e-12
assert product_target_fidelity > 1 - 1e-12


target logical theta                = 0.196349540849 rad
physical STAR theta_star           = -0.865268077594 rad
success probability                = 0.565351992731
ideal STAR probability             = 0.565351992731
fidelity vs direct encoded target  = 1.0000000000000000
fidelity vs product-rotation check = 1.0000000000000000


In [3]:
# Show why postselection works: among the 2^3 terms from the three physical Rz gates,
# only the identity term and the full logical-Z term have zero X syndrome.
from itertools import combinations


def x_syndrome_for_z_error(z_error_support: tuple[int, ...]) -> tuple[int, ...]:
    return tuple((len(set(z_error_support) & set(check)) & 1) for check in STEANE_CHECKS)


for r in range(len(STAR_LOGICAL_Z_SUPPORT) + 1):
    for subset in combinations(STAR_LOGICAL_Z_SUPPORT, r):
        syndrome = x_syndrome_for_z_error(subset)
        accepted_term = syndrome == (0, 0, 0)
        print(f"Z{subset!s:<10} syndrome={syndrome} accepted={accepted_term}")


Z()         syndrome=(0, 0, 0) accepted=True
Z(4,)       syndrome=(0, 1, 1) accepted=False
Z(5,)       syndrome=(0, 0, 1) accepted=False
Z(6,)       syndrome=(0, 1, 0) accepted=False
Z(4, 5)     syndrome=(0, 1, 0) accepted=False
Z(4, 6)     syndrome=(0, 0, 1) accepted=False
Z(5, 6)     syndrome=(0, 1, 1) accepted=False
Z(4, 5, 6)  syndrome=(0, 0, 0) accepted=True


## Noiseless k-sweep

Here `k` is the number of physical Z-string rotations in the TMR layer. For the same Steane logical support `Z4 Z5 Z6`:

- `k=1`: one product rotation `Rzzz` on `(4, 5, 6)`.
- `k=2`: one `Rzz` on `(4, 5)` and one `Rz` on `(6,)`.
- `k=3`: three single-qubit rotations `Rz` on `(4,)`, `(5,)`, `(6,)`.

The raw fidelity column compares the accepted state to the directly encoded target `Rz(theta)|+>_L`. The corrected fidelity column applies the needed logical Clifford frame correction. For `k=2`, that correction is `Rx_L(pi/2)`, which converts the raw `(-i)^2=-1` relative phase into the desired `-i` phase.


In [4]:
theta = math.pi / 16
protocols = {
    1: ((4, 5, 6),),
    2: ((4, 5), (6,)),
    3: ((4,), (5,), (6,)),
}

print(" k | supports              | theta_star(rad) | success      | raw fid vs Rz | corrected fid")
print("---+-----------------------+-----------------+--------------+---------------+---------------")
for k, supports in protocols.items():
    (
        theta_star,
        p_success,
        direct_target_fidelity,
        product_target_fidelity,
        accepted,
        direct_target,
    ) = star_inject_and_postselect(theta, supports)

    corrected = accepted
    correction = "none"
    if k == 2:
        corrected = normalize(apply_logical_rx(accepted, STAR_LOGICAL_X_SUPPORT, math.pi / 2))
        correction = "Rx_L(pi/2)"

    corrected_fidelity = abs(inner(direct_target, corrected)) ** 2
    support_label = " ".join(str(s) for s in supports)
    print(
        f"{k:2d} | {support_label:<21} | "
        f"{theta_star:15.12f} | "
        f"{p_success:12.9f} | "
        f"{direct_target_fidelity:13.10f} | "
        f"{corrected_fidelity:13.10f}   {correction}"
    )

assert corrected_fidelity > 1 - 1e-12


 k | supports              | theta_star(rad) | success      | raw fid vs Rz | corrected fid
---+-----------------------+-----------------+--------------+---------------+---------------
 1 | (4, 5, 6)             |  0.196349540849 |  1.000000000 |  1.0000000000 |  1.0000000000   none
 2 | (4, 5) (6,)           |  0.608198354993 |  0.836756839 |  0.9809698831 |  1.0000000000   Rx_L(pi/2)
 3 | (4,) (5,) (6,)        | -0.865268077594 |  0.565351993 |  1.0000000000 |  1.0000000000   none


## Physical Squin kernel

This is the physical-kernel version of the STAR layer. It allocates one Steane block at explicit Gemini physical sites, initializes it as `|+>_L`, applies the three physical STAR rotations, and then measures the seven physical qubits.

The statevector cells above are the ideal protocol verifier. A full hardware-style nondestructive syndrome-extraction version would add Steane ancilla checks around this STAR layer, then postselect on the measured `X`-check syndrome.

In [5]:
try:
    from typing import Any, Literal

    from bloqade import squin
    from bloqade.gemini.common.dialects import qubit
    from bloqade.lanes.arch.gemini.logical import steane7_initialize
    from bloqade.lanes.passes import ASAPPlacePass
    from bloqade.lanes.pipeline import PhysicalPipeline
    from bloqade.types import Qubit
    from kirin.dialects import ilist
    from kirin.dialects.ilist import IList

    BLOQADE_AVAILABLE = True
except ModuleNotFoundError as exc:
    BLOQADE_AVAILABLE = False
    print(f"Bloqade imports unavailable in this kernel: {exc}")


if BLOQADE_AVAILABLE:
    kernel = squin.kernel.add(qubit)
    kernel.run_pass = squin.kernel.run_pass

    LogicalQubit = ilist.IList[Qubit, Literal[7]]

    def slot_allocator():
        slot_words = ilist.IList([0, 4, 8, 12, 16, 2, 6, 10, 14, 18])
        slots = IList(
            [
                IList([(0, word_id, site_id) for site_id in range(7)])
                for word_id in slot_words
            ]
        )

        @kernel
        def qalloc_slot(slot_index: int, theta: float, phi: float, lam: float) -> LogicalQubit:
            def allocate_at(address: tuple[int, int, int]):
                return qubit.new_at(address[0], address[1], address[2])

            reg = ilist.map(allocate_at, slots[slot_index])
            steane7_initialize(theta, phi, lam, reg)
            return reg

        return qalloc_slot

    qalloc_slot = slot_allocator()

    @kernel(typeinfer=True)
    def star_inject_steane7_physical(theta_star: float):
        # qalloc_slot(..., pi/2, 0, 0) prepares the injected physical qubit as |+>
        # before the Steane encoder, giving |+>_L.
        block = qalloc_slot(0, math.pi / 2, 0.0, 0.0)

        # STAR/TMR layer on the weight-3 logical-Z support [4, 5, 6].
        squin.rz(theta_star, block[4])
        squin.rz(theta_star, block[5])
        squin.rz(theta_star, block[6])

        return squin.broadcast.measure(block)

    print("Defined physical Squin kernel: star_inject_steane7_physical(theta_star)")


Defined physical Squin kernel: star_inject_steane7_physical(theta_star)


In [6]:
# Optional: compile the physical Squin kernel to move IR to inspect the physical placements.
# This is intentionally separate from the statevector verifier above.
if BLOQADE_AVAILABLE:
    theta_star_for_kernel = star_theta_star(math.pi / 16)
    move_mt = PhysicalPipeline(place_opt_type=ASAPPlacePass).emit(
        star_inject_steane7_physical
    )
    print(f"theta_star_for_kernel = {theta_star_for_kernel:.12f}")
    move_mt.print()


theta_star_for_kernel = -0.865268077594
func.func @star_inject_steane7_physical(theta_star : !py.float) -> !Bottom {
  ^0(%star_inject_steane7_physical_self, %theta_star):
  │       %0 = lanes.move.load() : !py.State
  │       %1 = lanes.move.fill(current_state=%0){location_addresses=(0x0000000000000000, 0x0000000001000000, 0x0000000002000000, 0x0000000003000000, 0x0000000004000000, 0x0000000005000000, 0x0000000006000000) : !py.tuple[!py.LocationAddress, ...]} : !py.State
  │            lanes.move.store(current_state=%1)
  │   %theta = py.constant.constant 0.25 : !py.float
  │     %phi = py.constant.constant 0.0 : !py.float
  │   %angle = py.constant.constant -0.25 : !py.float
  │ %angle_1 = py.constant.constant 0.5 : !py.float
  │       %2 = py.constant.constant 6.283185307179586 : !py.float
  │       %3 = lanes.move.local_rz(current_state=%1, rotation_angle=%phi){location_addresses=(0x0000000006000000,) : !py.tuple[!py.LocationAddress, ...]} : !py.State
  │            lanes.move.stor

## Noiseless Squin statevector verification

This cell simulates an unpinned physical Squin version of the same seven-qubit STAR circuit. The pinned `new_at` addresses are placement constraints for the move compiler, so the statevector simulator uses ordinary `squin.qalloc(7)` and the same `steane7_initialize` plus physical STAR rotations.

The check compares the postselected simulated state against `Rz_L(theta)|+>_L`.


In [7]:
if BLOQADE_AVAILABLE:
    from bloqade.pyqrack.device import StackMemorySimulator

    @squin.kernel(typeinfer=True)
    def steane7_plus_state():
        reg = squin.qalloc(7)
        steane7_initialize(math.pi / 2, 0.0, 0.0, reg)
        return reg

    @squin.kernel(typeinfer=True)
    def star_inject_steane7_state(theta_star: float):
        reg = squin.qalloc(7)
        steane7_initialize(math.pi / 2, 0.0, 0.0, reg)
        squin.rz(theta_star, reg[4])
        squin.rz(theta_star, reg[5])
        squin.rz(theta_star, reg[6])
        return reg

    theta = math.pi / 16
    theta_star = star_theta_star(theta)
    simulator = StackMemorySimulator()

    initial_squin_state = list(simulator.state_vector(steane7_plus_state))
    star_squin_state = list(
        simulator.state_vector(star_inject_steane7_state, args=(theta_star,))
    )

    accepted_squin_state = star_squin_state
    for check in STEANE_CHECKS:
        accepted_squin_state = project_x_plus(accepted_squin_state, check)

    squin_success_probability = norm2(accepted_squin_state)
    accepted_squin_state = normalize(accepted_squin_state)
    # Compare against the directly encoded target Rz(theta)|+>_L.
    target_squin_state = directly_encoded_rz_plus(theta)
    squin_fidelity = abs(inner(target_squin_state, accepted_squin_state)) ** 2

    print(f"target logical theta      = {theta:.12f} rad")
    print(f"physical STAR theta_star = {theta_star:.12f} rad")
    print(f"Squin success probability = {squin_success_probability:.12f}")
    print(f"ideal STAR probability    = {ideal_star_success_probability(theta_star):.12f}")
    print(f"postselected fidelity     = {squin_fidelity:.16f}")

    assert abs(squin_success_probability - ideal_star_success_probability(theta_star)) < 1e-6
    assert squin_fidelity > 1 - 1e-12
else:
    print("Skipping Squin statevector verification because Bloqade is unavailable.")


┌───────────────────────────────────────────────────────────────

│ INFO:

Begin Steane7 Initialize


└───────────────────────────────────────────────────────────────

┌───────────────────────────────────────────────────────────────

...

┌───────────────────────────────────────────────────────────────

│ INFO:

End Steane7 Initialize


└───────────────────────────────────────────────────────────────

target logical theta      = 0.196349540849 rad
physical STAR theta_star = -0.865268077594 rad
Squin success probability = 0.565351710862
ideal STAR probability    = 0.565351992731
postselected fidelity     = 1.0000000000000000


## Move visualization

Run this cell after compiling `move_mt`. With a Qt/Jupyter GUI backend, the debugger opens an interactive move viewer in a separate window.


In [8]:
if BLOQADE_AVAILABLE:
    from bloqade.lanes.arch.gemini.physical import get_arch_spec
    from bloqade.lanes.visualize import animated_debugger, debugger

    try:
        get_ipython().run_line_magic("matplotlib", "qt")
    except Exception as exc:
        print(f"Could not switch to Qt matplotlib backend: {exc}")

    if "move_mt" not in globals():
        move_mt = PhysicalPipeline(place_opt_type=ASAPPlacePass).emit(
            star_inject_steane7_physical
        )

    # Static step-through viewer. Use animated_debugger(..., fps=30) for smooth movement.
    debugger(move_mt, get_arch_spec(), interactive=True, atom_marker="o")
    # animated_debugger(move_mt, get_arch_spec(), interactive=True, atom_marker="o", fps=30)
else:
    print("Skipping move visualization because Bloqade is unavailable.")
